In [27]:
import numpy as np

class RNN():

    def __init__(self):

        self.hidden_state = np.zeros(3)

        self.W_hh = np.random.randn(3,3)
        self.W_xh = np.random.randn(3,3)
        self.W_hy = np.random.randn(3,3)

        self.Bh = np.random.randn(3)
        self.By = np.random.randn(3)

    def forward(self, x, target):

        # Save previous hidden state
        self.h_prev = self.hidden_state.copy()

        # Hidden state
        self.hidden_state = np.tanh(
            self.h_prev.dot(self.W_hh)
            + x.dot(self.W_xh)
            + self.Bh
        )

        # Output
        self.y = self.hidden_state.dot(self.W_hy) + self.By

        # Softmax predict probability
        exp = np.exp(self.y - np.max(self.y))
        self.sm = exp / np.sum(exp)

        # Cross entropy loss
        self.loss = -np.sum(target * np.log(self.sm + 1e-8))

        return self.y, self.sm, self.loss

    def backward(self, x, target, lr=0.01):
   
        dy = self.sm - target  # dL/dy

        # Output layer gradients

        #self.hidden_state=np.reshape(-1,1)
        #dy=np.reshape(1,-1)
        #dWhy=np.dot(self.hidden_state,dy)
        dWhy = np.outer(self.hidden_state, dy) #h-here reshaping we cant transpose because 1d array

        dBy = dy

        # Hidden layer gradient
        dh = dy.dot(self.W_hy.T)

        # tanh derivative
        dtanh = dh * (1 - self.hidden_state ** 2)


        # Hidden weights
        dWxh = np.outer(x, dtanh)

        dWhh = np.outer(self.h_prev, dtanh)

        dBh = dtanh


        # Gradient Descent Update
        self.W_hy -= lr * dWhy
        self.By -= lr * dBy

        self.W_xh -= lr * dWxh
        self.W_hh -= lr * dWhh

        self.Bh -= lr * dBh

    def predict(self, x):

        # Hidden state update
        self.hidden_state = np.tanh(
            self.hidden_state.dot(self.W_hh)
            + x.dot(self.W_xh)
            + self.Bh
        )

        # Output layer
        y = self.hidden_state.dot(self.W_hy) + self.By

        # Softmax
        exp = np.exp(y - np.max(y))
        prob = exp / np.sum(exp)


        return prob
        

In [28]:
model = RNN()

x = np.array([1,1,1])

target = np.array([1,0,0])


# Training
for epoch in range(1000):

    y, prob, loss = model.forward(x,target)

    model.backward(x,target)

    if epoch % 100 == 0:
        print("epoch:",epoch,"loss:",loss)



epoch: 0 loss: 1.1839456110256756
epoch: 100 loss: 0.1510532065866335
epoch: 200 loss: 0.08059961685080148
epoch: 300 loss: 0.05450270711596528
epoch: 400 loss: 0.041076032150591184
epoch: 500 loss: 0.03292831994687387
epoch: 600 loss: 0.02746730397554469
epoch: 700 loss: 0.023555575665354864
epoch: 800 loss: 0.02061714621222091
epoch: 900 loss: 0.01832959515523968


In [29]:
test_input = np.array([1,1,1])
probability = model.predict(test_input)
print("Probability:", probability)

Probability: [0.9836368  0.00592707 0.01043613]
